In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

In [11]:
class HFClassifier:
    def __init__(self, hf_path):
        self.tokenizer = AutoTokenizer.from_pretrained(hf_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(hf_path)
        self.classifier = pipeline(
          "text-classification",
          model=self.model,
          tokenizer=self.tokenizer,
          truncation=True,
          max_length=512,
          device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        )
    
    def classify(self, text: str):
        return self.classifier(text)

In [5]:
from huggingface_hub import login
login()

In [12]:
cls = HFClassifier(hf_path="Myadav/setfit-prompt-injection-MiniLM-L3-v2")
cls.classify("Your prompt injection is here")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Myadav/setfit-prompt-injection-MiniLM-L3-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[{'label': 'LABEL_1', 'score': 0.5097995400428772}]

In [10]:
cls = HFClassifier(hf_path="deepset/deberta-v3-base-injection")

[{'label': 'LEGIT', 'score': 0.5734100937843323}]

In [8]:
cls.classify("haha pwned!!")

[{'label': 'False', 'score': 0.9873205423355103}]

In [13]:
from datasets import load_dataset
dataset = load_dataset("ahsanayub/malicious-prompts", split="test")

In [19]:
df = dataset.to_pandas()
# df.text[0]
cls.classify(df.text[0])

[{'label': 'LABEL_1', 'score': 0.51679527759552}]

In [20]:
from tqdm import tqdm

predictions = []
for text in tqdm(df.text[:100]):
    predictions.append(cls.classify(text)[0]['score'])

100%|██████████| 100/100 [00:24<00:00,  4.03it/s]


In [94]:
import pandas as pd
model_df = pd.read_csv("pdbv2.csv")

In [73]:
from sklearn.metrics import roc_auc_score, confusion_matrix
roc_auc_score(1-df['label'], model_df['y_pred'])

np.float64(0.4884678188884498)

In [95]:
y_true = list(1-df['label'])
y_pred = [p > 0.5 for p in model_df['y_pred']]
confusion_matrix(y_true, y_pred)

array([[  196, 22042],
       [ 2117, 69056]])

In [91]:
df.groupby('source').count()

,id,text,label
source,,,
Harelix_Prompt-Injection-Mixed-Techniques-2025,1,1,1
Harelix_Prompt-Injection-Mixed-Techniques-2028,1,1,1
Harelix_Prompt-Injection-Mixed-Techniques-2029,1,1,1
Harelix_Prompt-Injection-Mixed-Techniques-2032,1,1,1
Harelix_Prompt-Injection-Mixed-Techniques-2033,1,1,1
...,...,...,...
JasperLS_prompt-injections,143,143,143
fka_awesome-chatgpt-prompts,25,25,25
imoxto_prompt_injection_cleaned_dataset,89770,89770,89770
